# Persona-aware Context — one agent, two users, two realities

The **same** agent, the **same** system prompt, the **same** tools — run twice. The only
thing that changes is *which user's token* authenticates the MCP calls. Every difference
below is produced **server-side by OpenMetadata**, never by the prompt.

|  | David Kim | Sara Johnson |
|--|-----------|--------------|
| **Persona** | `ComplianceOfficer` | `DataEngineer` |
| **Persona context** | tags · glossary · articles · data quality | schema · constraints · joins · lineage · profile |
| **Operating rules** | cite the policy, name the sensitive columns | dbt path, freshness SLA, runnable `SELECT` |
| **Same `dim_customers` asset** | PII profiles **visible** (he owns it) | PII profiles **withheld** |

Two capabilities, both scoped live by the caller's identity:

- **`get_persona_context`** — a curated, per-persona working document. Same catalog, different lens.
- **`get_entity_details(include=["context"])`** — one asset's full Context Profile,
  RBAC-filtered per caller. (Earlier notes call this tool `get_asset_context`; on an
  OpenMetadata 2.0 server it is a section of `get_entity_details`.)

```
        Same code · same prompt · same question
                         │
       ┌─────────────────┴──────────────────┐
   David Kim's token                 Sara Johnson's token
   (ComplianceOfficer)                 (DataEngineer)
       │      client.mcp — identical wiring       │
       └─────────────────┬──────────────────┘
                         ▼
                 OpenMetadata MCP
        get_persona_context  → persona lens
        get_entity_details   → RBAC / PII masking
          include=[context]
```

> **Prerequisites:** the [Jaffle Shop demo database](../resources/demo-database/) ingested
> into an **OpenMetadata 2.0** instance, then `setup_demo.py` run once with an admin token —
> it creates the users, the personas, the PII tags, the profiles, the data-quality results,
> and prints one access token per user. See [README.md](./README.md). The raw-tool proof
> needs only `data-ai-sdk`; the optional agent cells also need `data-ai-sdk[langchain]`
> and an LLM key.

## 1. Two identities, two tokens

Point the notebook at your instance and drop in one token per user. Nothing here is
persona-specific yet — just *who is calling*. Using two real user tokens (rather than one
admin/bot token) is what makes RBAC actually apply: admins and bots see everything.

In [ ]:
!pip install ../../python

In [1]:
import os
from pathlib import Path

# Secrets live in a gitignored .env next to this notebook — never commit them.
# Copy .env.example to .env and fill in your values (one KEY=value per line).
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    print(f"Loaded secrets from {env_path.resolve()}")
else:
    print(".env not found — copy .env.example to .env and fill in your values.")

Loaded secrets from /Users/pmbrull/conductor/workspaces/ai-sdk/athens/cookbook/persona-aware-context/.env


In [2]:
import os
from dataclasses import dataclass

from ai_sdk import AISdk
from ai_sdk.mcp.models import MCPTool


@dataclass
class Identity:
    """A demo user: a label, the persona they carry, and their access token."""

    label: str
    persona: str
    token: str


HOST = os.environ.get("AI_SDK_HOST", "http://localhost:8585").rstrip("/")

# One access token per user — `setup_demo.py` prints both, or generate them in
# OpenMetadata: log in as each user -> profile -> Access Tokens.
COMPLIANCE_TOKEN = os.environ.get("DEMO_COMPLIANCE_TOKEN", "")
ENGINEER_TOKEN = os.environ.get("DEMO_ENGINEER_TOKEN", "")

missing = [
    name
    for name, value in (
        ("DEMO_COMPLIANCE_TOKEN", COMPLIANCE_TOKEN),
        ("DEMO_ENGINEER_TOKEN", ENGINEER_TOKEN),
    )
    if not value
]
if missing:
    raise SystemExit(
        "Missing environment variable(s): "
        + ", ".join(missing)
        + "\nRun setup_demo.py — it prints one token per demo user (see README.md)."
    )

compliance = Identity("David Kim", "ComplianceOfficer", COMPLIANCE_TOKEN)
engineer = Identity("Sara Johnson", "DataEngineer", ENGINEER_TOKEN)

compliance_client = AISdk(host=HOST, token=compliance.token)
engineer_client = AISdk(host=HOST, token=engineer.token)

print(f"Host       : {HOST}")
print(f"Compliance : {compliance.label:<13} -> {compliance.persona}")
print(f"Engineer   : {engineer.label:<13} -> {engineer.persona}")

Host       : http://localhost:8585
Compliance : David Kim     -> ComplianceOfficer
Engineer   : Sara Johnson  -> DataEngineer


## 2. The shared setup — identical for both users

One tool list. One system prompt with **nothing persona-specific in it**. The agent is
told to trust the tools; the tools are already scoped to the caller. That is what makes
the two runs diverge.

In [3]:
# The read-only, context-focused tool set both identities receive. Identical for
# everyone — the server decides what each call is allowed to return.
#
# `get_entity_details(include=["context"])` is how an OpenMetadata 2.0 server
# returns an asset's Context Profile; older notes call that tool
# `get_asset_context`. Intersecting with what the server actually advertises
# keeps this cell working across builds instead of failing with
# "Unknown tool: invalid_tool_name".
WANTED_TOOLS: list[MCPTool] = [
    MCPTool.GET_PERSONA_CONTEXT,
    MCPTool.GET_ENTITY_DETAILS,
    MCPTool.FIND_CONTEXT,
    MCPTool.SEARCH_METADATA,
    MCPTool.SEMANTIC_SEARCH,
    MCPTool.GET_ENTITY_LINEAGE,
]
available = {tool.name for tool in compliance_client.mcp.list_tools()}
DEMO_TOOLS: list[MCPTool] = [tool for tool in WANTED_TOOLS if tool in available]
print("tools:", ", ".join(tool.value for tool in DEMO_TOOLS))
missing_tools = [tool.value for tool in WANTED_TOOLS if tool not in available]
if missing_tools:
    print("not advertised by this server:", ", ".join(missing_tools))

# One prompt for everyone. Nothing persona-specific here: the agent trusts the tools,
# and the tools are already scoped to the caller. That is what makes the two runs diverge.
SYSTEM_PROMPT = """You are a data catalog assistant embedded in OpenMetadata.

You answer using ONLY the tools provided. Those tools already return context
scoped to the current user's persona and permissions — you never choose a
persona or filter anything yourself.

Tool guide:
- ALWAYS begin by calling get_persona_context with no arguments — this returns
  YOUR persona's working context and OPERATING RULES (how you should answer).
  Do this first for EVERY question, including questions about a specific asset.
- Then, for a specific asset, call get_entity_details with entityType, fqn and
  include=["context", "quality"] — that returns the asset's Context Profile and
  its data-quality standing. If you only know the asset's name, call
  search_metadata first to resolve its fully qualified name.
- Use find_context for glossary definitions and documentation.

Rules:
- Your persona context may contain OPERATING RULES (for example a knowledge
  article titled "... Operating Rules"). Treat them as instructions for HOW to
  answer: what to lead with, what to cite, what format to use. Apply them and
  state which rule you followed.
- Never invent columns, tags, metrics, or values that are not in the tool
  output. If a column is masked, redacted, or absent, say so explicitly.
- Be faithful and concise. Report what your persona and permissions actually
  expose, and call out anything that appears restricted."""

PERSONA_QUESTION = (
    "What's my working context? Summarize what I should focus on for our "
    "customer and order data, and why those things matter for my role."
)

ASSET_QUESTION_TEMPLATE = (
    "Give me the full context for the `{table}` table: its columns, any "
    "sensitive/PII fields, profiling detail if available, and its data-quality "
    "standing. Be explicit about anything you cannot see."
)

# The PII asset both users will inspect. Override DEMO_TABLE_FQN if your service
# name differs — this value matches the Jaffle Shop demo database.
TABLE = "dim_customers"
TABLE_FQN = os.environ.get(
    "DEMO_TABLE_FQN", "jaffle shop.jaffle_shop.marts_core.dim_customers"
)

tools: get_persona_context, get_entity_details, find_context, search_metadata, semantic_search, get_entity_lineage


## 3. Helpers — raw MCP calls + side-by-side rendering

Two thin wrappers around `client.mcp.call_tool(...)` (no LLM — deterministic proof) and
two display helpers so the difference is easy to *see* on a screen.

In [4]:
import difflib
import html

from IPython.display import HTML, Markdown, display


def _extract_markdown(data: object) -> str:
    """Pull the rendered markdown out of an MCP tool result payload."""
    if isinstance(data, dict):
        for key in ("content", "markdown", "text"):
            value = data.get(key)
            if isinstance(value, str) and value.strip():
                return value
    if isinstance(data, str):
        return data
    return str(data)


def raw_persona_context(client: AISdk) -> str:
    """The caller's persona working document as markdown (no LLM)."""
    result = client.mcp.call_tool(MCPTool.GET_PERSONA_CONTEXT, {"format": "markdown"})
    if not result.success or result.data is None:
        return f"(no persona context returned: {result.error})"
    return _extract_markdown(result.data)


def raw_asset_context(client: AISdk, fqn: str) -> str:
    """One asset's Context Profile as markdown, RBAC-filtered for the caller (no LLM)."""
    result = client.mcp.call_tool(
        MCPTool.GET_ENTITY_DETAILS,
        {
            "entityType": "table",
            "fqn": fqn,
            "include": ["context", "quality"],
            "format": "markdown",
        },
    )
    if not result.success or result.data is None:
        return f"(no asset context returned: {result.error})"
    payload = result.data
    section = payload.get("context") if isinstance(payload, dict) else None
    return _extract_markdown(section if section is not None else payload)


def _strip_timestamp(document: str) -> str:
    """Drop the generated-at line so a diff shows content, not clock skew."""
    return "\n".join(
        line
        for line in document.splitlines()
        if not line.startswith(("timestamp:", "generated_at:", "fingerprint:"))
    )


def show_side_by_side(left: str, right: str, left_label: str, right_label: str) -> None:
    """Render two markdown documents in side-by-side columns (raw source, monospace)."""
    column = (
        "<div style='flex:1;min-width:0;border:1px solid #d0d7de;border-radius:6px;overflow:hidden'>"
        "<div style='background:{bg};color:#fff;padding:6px 10px;font-weight:600;font-family:sans-serif'>{label}</div>"
        "<pre style='margin:0;padding:10px;white-space:pre-wrap;font-size:12px;line-height:1.45;max-height:520px;overflow:auto'>{body}</pre>"
        "</div>"
    )
    left_html = column.format(bg="#8250df", label=html.escape(left_label), body=html.escape(left))
    right_html = column.format(bg="#1f6feb", label=html.escape(right_label), body=html.escape(right))
    display(HTML(f"<div style='display:flex;gap:12px;align-items:stretch'>{left_html}{right_html}</div>"))


def show_diff(left: str, right: str, left_label: str, right_label: str) -> None:
    """Unified diff so withheld / omitted lines are obvious."""
    body = "\n".join(
        difflib.unified_diff(
            _strip_timestamp(left).splitlines(),
            _strip_timestamp(right).splitlines(),
            fromfile=left_label,
            tofile=right_label,
            lineterm="",
        )
    )
    if not body.strip():
        display(Markdown("**Identical** — did `setup_demo.py` apply the PII tags and ownership?"))
        return
    display(Markdown(f"```diff\n{body}\n```"))
    display(
        Markdown(
            f"Lines prefixed `-` are visible only to **{left_label}**; `+` only to **{right_label}**."
        )
    )

## Scene 1 — AI Persona Context: "what should I focus on?"

Both users ask the identical question. `get_persona_context` resolves each caller's active
persona and returns *their* curated working document. Same catalog, two lenses — no LLM
involved, this is the raw MCP response, side by side.

Look for three differences: **which sections render** (compliance gets tags, glossary,
articles and data quality; engineering gets schema, constraints, joins, lineage, profile),
**the operating-rules article** each persona pulls in, and the **shared knowledge** block at
the end of the compliance document.

> The persona document is a **cached** render (`cacheTtlMinutes`, set to 5 by `setup_demo.py`)
> built from indexed metadata, so its `Data Quality` counts can lag the catalog. The live
> figure — 4 passed, 1 failed — is in the asset Context Profile in Scene 2, which is read
> straight through. If a section here looks stale, that is what you are seeing.

In [5]:
show_side_by_side(
    raw_persona_context(compliance_client),
    raw_persona_context(engineer_client),
    f"{compliance.label} · {compliance.persona}",
    f"{engineer.label} · {engineer.persona}",
)

### Same question through the agent

The block above is the deterministic proof — no LLM. Below, an **identical** LangChain
agent answers `PERSONA_QUESTION` under each identity. Needs `data-ai-sdk[langchain]` and an
LLM key (`OPENAI_API_KEY` / `ANTHROPIC_API_KEY`). `USE_AGENT` auto-detects a key; set it to
`False` to skip.

```
PERSONA_QUESTION = (
    "What's my working context? Summarize what I should focus on for our "
    "customer and order data, and why those things matter for my role."
)
```

In [6]:
USE_AGENT = bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))
MODEL = os.environ.get("DEMO_MODEL", "openai:gpt-4o")


def build_agent(client: AISdk, model: str):
    """An identical LangChain agent wired to the caller's scoped MCP tools.

    ``as_langchain_tools`` lists tools under this client's token, so two clients
    yield two tool sets that behave differently from identical code.
    """
    from langchain.agents import create_agent

    return create_agent(
        model=model,
        tools=client.mcp.as_langchain_tools(include=DEMO_TOOLS),
        system_prompt=SYSTEM_PROMPT,
    )


def ask(agent, question: str) -> str:
    """Invoke an agent and return its final text answer."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


if USE_AGENT:
    for identity, client in ((compliance, compliance_client), (engineer, engineer_client)):
        display(Markdown("-----"))
        display(Markdown(f"#### {identity.label} · {identity.persona}"))
        display(Markdown(ask(build_agent(client, MODEL), PERSONA_QUESTION)))
else:
    print("USE_AGENT is False — skipping the LLM. The side-by-side above is the proof.")

USE_AGENT is False — skipping the LLM. The side-by-side above is the proof.


## Scene 2 — AI Entity Context: the same asset, RBAC-filtered

Now both identities call `get_asset_context` on the **same** `dim_customers`, with the same
arguments. `setup_demo.py` tagged the sensitive columns `PII.Sensitive` and made David an
**owner**; Sara holds the same `DataConsumer` role but does not own the asset.

OpenMetadata drops the profile of a `PII.Sensitive` column for anyone who is not an admin,
a bot, or an owner — **server-side, before the response is serialised**. The agent Sara is
talking to never receives those values, so no amount of prompting can make it reveal them.

In [7]:
david_view = raw_asset_context(compliance_client, TABLE_FQN)
sara_view = raw_asset_context(engineer_client, TABLE_FQN)

show_diff(
    david_view,
    sara_view,
    f"{compliance.label} · owner",
    f"{engineer.label} · non-owner",
)

```diff
--- David Kim · owner
+++ Sara Johnson · non-owner
@@ -207,14 +207,8 @@
 | Column | Null % | Distinct | Min | Max |
 |--------|--------|----------|-----|-----|
 | customer_id | 0% | 25 | 1.0 | 25.0 |
-| first_name | 4% | 23 |  |  |
-| last_name | 0% | 24 |  |  |
-| full_name | 4% | 23 |  |  |
-| email | 8% | 22 |  |  |
-| phone_number | 4% | 23 |  |  |
 | city | 4% | 22 |  |  |
 | state | 4% | 19 |  |  |
-| postal_code | 4% | 22 |  |  |
 | country | 0% | 1 |  |  |
 | customer_created_at | 0% | 24 |  |  |
 | total_orders | 0% | 8 | 1.0 | 9.0 |
```

Lines prefixed `-` are visible only to **David Kim · owner**; `+` only to **Sara Johnson · non-owner**.

Same call, same table, two payloads. Note *what* disappears: the schema still lists
`email` and `phone_number` — Sara can see the columns exist and what type they are — but
their **profile rows are gone**. Structure is metadata; distributions and value shapes are
data.

In [8]:
def profile_columns(document: str) -> set[str]:
    """Column names that appear in the document's Data Profile table.

    Scoped to that one section on purpose: the Schema table lists every column
    for both callers, so a naive substring search would report no difference.
    """
    names: set[str] = set()
    inside = False
    for line in document.splitlines():
        if line.startswith("# "):
            inside = line.strip() == "# Data Profile"
            continue
        if not inside or not line.startswith("|") or "---" in line:
            continue
        cell = line.split("|")[1].strip()
        if cell and cell != "Column":
            names.add(cell)
    return names


david_columns = profile_columns(david_view)
sara_columns = profile_columns(sara_view)
withheld = sorted(david_columns - sara_columns)

print(f"David Kim    (owner)     : {len(david_columns)} columns profiled")
print(f"Sara Johnson (non-owner) : {len(sara_columns)} columns profiled")
print()
print("Withheld from Sara :", ", ".join(withheld))
print(
    "Still in her schema:",
    all(f"| {name} |" in sara_view for name in withheld),
    "— she sees the columns exist, not their distributions",
)

David Kim    (owner)     : 26 columns profiled
Sara Johnson (non-owner) : 20 columns profiled

Withheld from Sara : email, first_name, full_name, last_name, phone_number, postal_code
Still in her schema: True — she sees the columns exist, not their distributions


### Sample rows — masked wholesale

The diff above is the **markdown** render, which has no Sample Data section. Ask for
`format="json"` and the Context Profile also carries actual rows under
`assetContext.table.sampleData` — and that is where the sharpest contrast lives.

Column profiles are filtered *per column*: Sara loses the six `PII.Sensitive` ones and keeps
the rest. Sample rows are filtered *wholesale*: one sensitive column redacts **every** value in
**every** row, because a row is only as shareable as its most sensitive field.

In [9]:
def sample_rows(client: AISdk, fqn: str) -> dict:
    """The sampleData block of an asset's Context Profile, as JSON."""
    result = client.mcp.call_tool(
        MCPTool.GET_ENTITY_DETAILS,
        {"entityType": "table", "fqn": fqn, "include": ["context"], "format": "json"},
    )
    if not result.success or not isinstance(result.data, dict):
        return {}
    # With format="markdown" the section arrives wrapped as {format, content};
    # with format="json" it IS the AIContext object. Handle both.
    context = result.data.get("context") or {}
    payload = context.get("content") if "content" in context else context
    table = ((payload or {}).get("assetContext") or {}).get("table") or {}
    return table.get("sampleData") or {}


for identity, client in ((compliance, compliance_client), (engineer, engineer_client)):
    sample = sample_rows(client, TABLE_FQN)
    columns = sample.get("columns") or []
    rows = sample.get("rows") or []
    masked = sum("[MASKED]" in name for name in columns)
    display(Markdown(f"**{identity.label}** — {masked}/{len(columns)} columns masked"))
    if not rows:
        print("(no sample rows returned)")
        continue
    keep = list(range(min(5, len(columns))))
    print(" | ".join(f"{columns[i][:20]:20}" for i in keep))
    for row in rows[:3]:
        print(" | ".join(f"{str(row[i])[:20]:20}" for i in keep))
    print()

**David Kim** — 0/26 columns masked

customer_id          | first_name           | last_name            | full_name            | email               
1                    | Michael              | Perez                | Michael Perez        | mperez@example.com  
2                    | Shawn                | Myers                | Shawn Myers          | smyers@example.com  
3                    | Kathleen             | Johnson              | Kathleen Johnson     | kjohnson@example.com



**Sara Johnson** — 26/26 columns masked

customer_id [MASKED] | first_name [MASKED]  | last_name [MASKED]   | full_name [MASKED]   | email [MASKED]      
********             | ********             | ********             | ********             | ********            
********             | ********             | ********             | ********             | ********            
********             | ********             | ********             | ********             | ********            

